# Preparação

Este documento é o *par org* do notebook `01-panorama-escolas.ipynb`. Os dois
são gerados um do outro por `make sync`; edite o que preferir, mas rode o sync
antes de commitar.

Nenhuma lógica de análise mora aqui — tudo vem do pacote `censo_escolar`. Se
você sentir vontade de escrever uma função dentro de um bloco, ela provavelmente
pertence a `src/censo_escolar/`.

In [ ]:
import pandas as pd

from censo_escolar import (
    carregar_escolas,
    colunas_disponiveis,
    panorama_por_uf,
    distribuicao_dependencia,
    taxa_infraestrutura,
    maiores_municipios,
    rotular,
)
from censo_escolar.loading import COLUNAS_BASICAS, COLUNAS_INFRA, COLUNAS_QUANTITATIVAS
from censo_escolar import plots

plots.aplicar_estilo()
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

ANO = 2023

## Que colunas existem neste ano?

O Censo renomeia e aposenta variáveis entre edições, então vale conferir antes
de fixar uma lista.

In [ ]:
colunas = colunas_disponiveis(ANO)
len(colunas), [c for c in colunas if c.startswith("QT_MAT")][:10]

# Carga

In [ ]:
escolas = carregar_escolas(
    ANO,
    colunas=[*COLUNAS_BASICAS, *COLUNAS_QUANTITATIVAS, *COLUNAS_INFRA],
)
escolas.shape

In [ ]:
escolas.head()

# Panorama por UF

In [ ]:
uf = panorama_por_uf(escolas)
uf.head(10)

No Jupyter o gráfico aparece sozinho. Aqui no org, `plots.barras` grava o PNG e
devolve o caminho; com `:results file` o Emacs transforma esse caminho num link
de imagem no `#+RESULTS:`. A mesma chamada serve aos dois ambientes.

In [ ]:
plots.barras(
    uf.head(15),
    x="SG_UF",
    y="escolas",
    titulo=f"Escolas em atividade por UF — {ANO}",
    rotulo_y="escolas",
    arquivo="escolas_por_uf.png",
)

# Dependência administrativa

In [ ]:
distribuicao_dependencia(escolas)

In [ ]:
distribuicao_dependencia(escolas, por="SG_UF").head(10)

# Infraestrutura

As colunas `IN_*` são binárias, então a média é diretamente a proporção de
escolas que possuem o item.

In [ ]:
taxa_infraestrutura(escolas)

Quebrando por rede — é aqui que a desigualdade aparece:

In [ ]:
por_rede = taxa_infraestrutura(rotular(escolas, "TP_DEPENDENCIA"), por="DS_DEPENDENCIA")
por_rede[["DS_DEPENDENCIA", "IN_INTERNET", "IN_BIBLIOTECA", "IN_ESGOTO_REDE_PUBLICA"]]

# Municípios com mais matrículas

In [ ]:
maiores_municipios(escolas, n=15)

# Próximos passos                                                  :ideias:

- Série histórica com `carregar_anos([2019, 2021, 2023])` e `serie_historica`.
- Cruzar com população municipal do IBGE para taxa de escolas por habitante.
- Suplementos do ZIP (matrícula, docentes, turmas) — hoje só lemos o arquivo de
  escolas.